# PHYS690 — Lecture 10: Initial-Value ODEs
## Euler, midpoint, Runge–Kutta, error, and validation

**Primary physics example:** a damped harmonic oscillator.

Today begins Unit 3: **Numerical Methods for Physical Systems**. The emphasis is not just on obtaining a trajectory, but on establishing that a numerical trajectory is trustworthy. We will repeatedly ask:

1. What differential equation are we actually solving?
2. What approximation does the algorithm make at each step?
3. How does the error scale when we change the step size?
4. How do physical diagnostics help us identify a bad numerical solution?
5. How do production solvers let us request accuracy without choosing every internal step by hand?


## Learning goals

By the end of class you should be able to:

- rewrite a second-order equation of motion as a first-order system;
- implement fixed-step **Euler**, **midpoint/RK2**, and **RK4** integrators;
- distinguish **local truncation error** from **global error**;
- measure an observed order of convergence from a step-size study;
- use a physical diagnostic—here, energy and the damping envelope—to validate a trajectory;
- use `scipy.integrate.solve_ivp` as a production-quality reference while still understanding the fixed-step methods underneath it.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

# Keep plots readable and consistent with the rest of the course.
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True})


## 1. Physics model: the damped harmonic oscillator

A mass on a linear spring with viscous damping obeys

$$
 m\ddot{x} + b\dot{x} + kx = 0.
$$

It is convenient to divide by the mass and define

$$
\omega_0^2 = \frac{k}{m},
\qquad
2\beta = \frac{b}{m},
$$

so that

$$
\boxed{\ddot{x} + 2\beta\dot{x}+\omega_0^2 x = 0.}
$$

For the **underdamped** case, $\beta < \omega_0$, the motion oscillates with damped angular frequency

$$
\omega_d = \sqrt{\omega_0^2-\beta^2}.
$$

We will use $x(0)=x_0$ and $\dot{x}(0)=v_0$. This is an **initial-value problem (IVP)**: the differential equation plus the complete state at one initial time determines the later state.


### Convert the second-order ODE to a first-order system

Most general ODE solvers expect a system of first-order equations. Define the velocity as a second state variable,

$$
v = \dot{x}.
$$

Then

$$
\frac{d}{dt}
\begin{pmatrix}x\\v\end{pmatrix}
=
\begin{pmatrix}
v\\
-2\beta v-\omega_0^2 x
\end{pmatrix}
\equiv f(t,\mathbf y).
$$

The numerical object we propagate is therefore the state vector

$$
\mathbf y(t)=\begin{pmatrix}x(t)\\v(t)\end{pmatrix}.
$$

This pattern is general: a $q$-th order ODE can be rewritten as a system of $q$ first-order ODEs by promoting derivatives to state variables.


In [ ]:
# Physical parameters in convenient units.
omega0 = 2.0       # natural angular frequency [rad/s]
beta = 0.15        # damping rate [1/s], beta < omega0 -> underdamped
x0 = 1.0           # initial displacement
v0 = 0.0           # initial velocity

y0 = np.array([x0, v0], dtype=float)
t_span = (0.0, 12.0)


def oscillator_rhs(t, y, omega0=omega0, beta=beta):
    """Return dy/dt for the damped harmonic oscillator.

    Parameters
    ----------
    t : float
        Time. The system is autonomous, but solvers still pass t explicitly.
    y : array-like, shape (2,)
        State vector [x, v].
    omega0 : float
        Undamped natural angular frequency.
    beta : float
        Damping rate in x'' + 2 beta x' + omega0^2 x = 0.
    """
    x, v = y
    dxdt = v
    dvdt = -2.0 * beta * v - omega0**2 * x
    return np.array([dxdt, dvdt])


## 2. Analytic benchmark—and an important initial-condition check

For $\beta<\omega_0$, the general underdamped solution can be written

$$
x(t)=e^{-\beta t}\left[A\cos(\omega_d t)+B\sin(\omega_d t)\right].
$$

The first initial condition gives $A=x_0$. Differentiating and applying $v(0)=v_0$ gives

$$
B=\frac{v_0+\beta x_0}{\omega_d}.
$$

Therefore the correct analytic solution for the initial conditions used here is

$$
\boxed{
x(t)=e^{-\beta t}\left[
x_0\cos(\omega_d t)
+\frac{v_0+\beta x_0}{\omega_d}\sin(\omega_d t)
\right].}
$$

A common mistake is to write only $e^{-\beta t}\cos(\omega_d t)$. That expression has $\dot{x}(0)=-\beta x_0$, not $0$, unless $\beta=0$. This is exactly the sort of mismatch an analytic benchmark should catch.


In [ ]:
def oscillator_exact(t, x0=x0, v0=v0, omega0=omega0, beta=beta):
    """Analytic underdamped solution x(t), v(t) for arbitrary x0 and v0."""
    t = np.asarray(t, dtype=float)
    omega_d = np.sqrt(omega0**2 - beta**2)
    A = x0
    B = (v0 + beta * x0) / omega_d

    exp_factor = np.exp(-beta * t)
    cos_term = np.cos(omega_d * t)
    sin_term = np.sin(omega_d * t)

    x = exp_factor * (A * cos_term + B * sin_term)

    # Differentiate the analytic expression so that x and v are benchmarked
    # consistently against the same initial conditions.
    v = exp_factor * (
        -beta * (A * cos_term + B * sin_term)
        + omega_d * (-A * sin_term + B * cos_term)
    )
    return x, v

# Explicitly verify the benchmark initial conditions.
x_check, v_check = oscillator_exact(np.array([0.0]))
print(f"x_exact(0) = {x_check[0]:.6f} (target {x0})")
print(f"v_exact(0) = {v_check[0]:.6f} (target {v0})")


## 3. Euler's method: the local linear approximation

For a general first-order IVP

$$
\mathbf y'(t)=f(t,\mathbf y),
$$

Taylor expansion around $t_n$ gives

$$
\mathbf y(t_n+h)
=
\mathbf y(t_n)+h\mathbf y'(t_n)+\mathcal O(h^2).
$$

Replacing $\mathbf y'(t_n)$ by $f(t_n,\mathbf y_n)$ gives **forward Euler**:

$$
\boxed{\mathbf y_{n+1}=\mathbf y_n+h f(t_n,\mathbf y_n).}
$$

The one-step defect is $\mathcal O(h^2)$. Because roughly $1/h$ steps are taken over a fixed interval, those local errors accumulate to a **global error** of $\mathcal O(h)$. Euler is therefore a **first-order method**.


In [ ]:
def euler_step(f, t, y, h):
    """Advance one step with forward Euler."""
    return y + h * f(t, y)


def integrate_fixed_step(f, stepper, t_span, y0, h):
    """Integrate y' = f(t,y) with a fixed-step one-step method.

    The final step is shortened if needed so that the returned trajectory ends
    exactly at t_span[1]. This keeps endpoint comparisons fair when h does not
    divide the interval perfectly.
    """
    t0, tf = t_span
    times = [float(t0)]
    states = [np.asarray(y0, dtype=float).copy()]

    t = float(t0)
    y = np.asarray(y0, dtype=float).copy()
    while t < tf - 10 * np.finfo(float).eps:
        h_step = min(h, tf - t)
        y = np.asarray(stepper(f, t, y, h_step), dtype=float)
        t = t + h_step
        times.append(t)
        states.append(y.copy())

    return np.asarray(times), np.vstack(states)

# A first look at Euler.
demo_h = 0.05 # time step in seconds
t_euler, y_euler = integrate_fixed_step(
    oscillator_rhs, euler_step, t_span, y0, h=demo_h
)

x_exact_euler_grid, _ = oscillator_exact(t_euler)

fig, ax = plt.subplots()
ax.plot(t_euler, x_exact_euler_grid, label="analytic")
ax.plot(t_euler, y_euler[:, 0], "--", label="Euler, h={} s".format(demo_h))
ax.set(xlabel="time [s]", ylabel="displacement x", title="Forward Euler: first comparison")
ax.legend()
plt.show()


### What should we inspect before declaring success?

A trajectory that “looks sinusoidal” is not enough. For this problem we have several independent checks:

- **Initial conditions:** does the code start from the requested state?
- **Analytic benchmark:** what is $x_\mathrm{num}-x_\mathrm{exact}$?
- **Step-size sensitivity:** does the result converge when $h$ is reduced?
- **Physical envelope:** should the oscillation amplitude grow or decay?
- **Mechanical energy:** with damping, energy should decrease according to the equation of motion.

A research-quality numerical result becomes much more convincing when several of these checks agree.


## 4. Worked example: midpoint / RK2

Euler samples the slope only at the beginning of the interval. Higher-order Runge-Kutta methods sample additional slopes in order to approximate the trajectory's curvature without explicitly differentiating $f$.

The midpoint method is a useful first example because it changes only one idea: estimate the slope at the middle of the step instead of only at the beginning.

$$
\begin{aligned}
k_1 &= f(t_n,y_n),\\
y_\mathrm{mid} &= y_n+\frac{h}{2}k_1,\\
k_2 &= f\left(t_n+\frac{h}{2},\;y_\mathrm{mid}\right),\\
y_{n+1} &= y_n+h k_2.
\end{aligned}
$$

The global error is $\mathcal O(h^2)$. In code, the pattern is: compute one slope, use it to estimate a midpoint state, compute the midpoint slope, and then take the full step with that midpoint slope.

In [ ]:
def midpoint_step(f, t, y, h):
    """Advance one step with the explicit midpoint (RK2) method."""
    # First estimate the slope at the beginning of the interval.
    k1 = f(t, y)

    # Use that slope to predict the state halfway across the interval.
    y_mid = y + 0.5 * h * k1

    # Estimate the slope at the midpoint.
    k2 = f(t + 0.5 * h, y_mid)

    # Use the midpoint slope for the full step.
    return y + h * k2


# Quick sanity check: with h=0, the state should not change.
np.testing.assert_allclose(midpoint_step(oscillator_rhs, t_span[0], y0, 0.0), y0)

In [ ]:
# Compare the worked Midpoint/RK2 method with Euler at the same step size.
rk2_demo_h = 0.1

t_euler_demo, y_euler_demo = integrate_fixed_step(
    oscillator_rhs, euler_step, t_span, y0, rk2_demo_h
)
t_midpoint_demo, y_midpoint_demo = integrate_fixed_step(
    oscillator_rhs, midpoint_step, t_span, y0, rk2_demo_h
)

x_exact_euler_demo, _ = oscillator_exact(t_euler_demo)
x_exact_midpoint_demo, _ = oscillator_exact(t_midpoint_demo)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(t_euler_demo, x_exact_euler_demo, "k-", label="analytic")
axes[0].plot(t_euler_demo, y_euler_demo[:, 0], "o--", ms=3, label="Euler")
axes[0].set_title("Euler")
axes[0].set_xlabel("time [s]")
axes[0].set_ylabel("displacement x")
axes[0].legend()

axes[1].plot(t_midpoint_demo, x_exact_midpoint_demo, "k-", label="analytic")
axes[1].plot(t_midpoint_demo, y_midpoint_demo[:, 0], "o--", ms=3, label="Midpoint / RK2")
axes[1].set_title("Midpoint / RK2")
axes[1].set_xlabel("time [s]")
axes[1].legend()

fig.suptitle(f"Same step size, improved slope estimate: h = {rk2_demo_h:g} s")
fig.tight_layout()
plt.show()

## In-class Activity 1 — Implement classical RK4 (10 minutes)

Now complete the classical fourth-order Runge-Kutta method. It uses four slope estimates:

$$
\begin{aligned}
k_1 &= f(t_n,y_n),\\
k_2 &= f(t_n+h/2,y_n+hk_1/2),\\
k_3 &= f(t_n+h/2,y_n+hk_2/2),\\
k_4 &= f(t_n+h,y_n+hk_3),\\
y_{n+1} &= y_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Work from the equation above rather than copying an implementation from the internet.

**Checks before moving on:**

1. `rk4_step` must return an array with the same shape as `y`.
2. If `h=0`, it should return exactly `y`.
3. With a sufficiently small step, it should track the analytic oscillator solution.

In [ ]:
def rk4_step(f, t, y, h):
    """Advance one step with the classical fourth-order Runge-Kutta method."""
    k1 = f(t, y)

    # TODO: compute k2 at the first midpoint estimate.
    k2 = ...

    # TODO: compute k3 using k2 to form a second midpoint estimate.
    k3 = ...

    # TODO: compute k4 at the end of the interval using k3.
    k4 = ...

    # TODO: combine k1, k2, k3, k4 with the RK4 weights 1,2,2,1.
    return ...

### Instructor reference: compare methods with a known-good implementation

The next cell uses the worked midpoint implementation and a reference RK4 implementation so the lecture can continue even while your Activity 1 function is still being debugged. After you finish `rk4_step`, compare it step-for-step with the reference version.

In [ ]:

def rk4_step_ref(f, t, y, h):
    """Reference classical RK4 step used for the instructor demonstration."""
    k1 = f(t, y)
    k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
    k3 = f(t + 0.5 * h, y + 0.5 * h * k2)
    k4 = f(t + h, y + h * k3)
    return y + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


rk4_demo_h = 0.1
methods = {
    "Euler": euler_step,
    "Midpoint/RK2": midpoint_step,
    "RK4": rk4_step_ref,
}

trajectories = {}
for name, stepper in methods.items():
    t_num, y_num = integrate_fixed_step(
        oscillator_rhs, stepper, t_span, y0, h=rk4_demo_h
    )
    trajectories[name] = (t_num, y_num)

# Use a dense grid only for plotting the analytic benchmark.
t_dense = np.linspace(t_span[0], t_span[1], 600)
x_exact_dense, v_exact_dense = oscillator_exact(t_dense)

fig, ax = plt.subplots()
ax.plot(t_dense, x_exact_dense, label="analytic", linewidth=2)
for name, (t_num, y_num) in trajectories.items():
    ax.plot(t_num, y_num[:, 0], label=f"{name}, h={rk4_demo_h:g}", alpha=0.85)
ax.set(xlabel="time [s]", ylabel="displacement x", title="Fixed-step methods compared with an analytic benchmark")
ax.legend(ncol=2)
plt.show()

## 5. A physical diagnostic: mechanical energy

For unit mass, define

$$
E(t)=\frac12 v^2+\frac12\omega_0^2x^2.
$$

Differentiate and substitute the equation of motion:

$$
\frac{dE}{dt}
= v\dot v+\omega_0^2x\dot x
= v(-2\beta v-\omega_0^2x)+\omega_0^2xv
= -2\beta v^2\le 0.
$$

So the exact mechanical energy is **monotonically non-increasing**. This is a strong diagnostic: a numerical trajectory whose energy systematically grows is not reproducing the damping physics, even if its first few oscillations look plausible.


In [ ]:
def oscillator_energy(y, omega0=omega0):
    """Mechanical energy per unit mass for states [x, v]."""
    y = np.asarray(y)
    x = np.take(y, 0, axis=-1)
    v = np.take(y, 1, axis=-1)
    return 0.5 * v**2 + 0.5 * omega0**2 * x**2

fig, ax = plt.subplots()
for name, (t_num, y_num) in trajectories.items():
    ax.plot(t_num, oscillator_energy(y_num), label=name)

analytic_states = np.column_stack([x_exact_dense, v_exact_dense])
ax.plot(t_dense, oscillator_energy(analytic_states), "k--", label="analytic")
ax.set(xlabel="time [s]", ylabel="energy per unit mass", title="Energy is a physics-based numerical diagnostic")
ax.legend()
plt.show()

### Discussion prompt

Suppose two methods give similar $x(t)$ for the first two periods, but one shows slowly increasing energy while the other shows decreasing energy. Which result would you trust over a longer integration, and why?

This is a recurring theme in computational physics: **a physically meaningful invariant or monotonic quantity often reveals failure earlier than a trajectory plot does.**


## 6. Local error, global error, and measured convergence

For a method of global order $p$, a useful empirical model is

$$
\epsilon(h) \approx C h^p
$$

once $h$ is small enough to be in the asymptotic convergence regime but not so small that floating-point roundoff dominates.

Taking logarithms,

$$
\log \epsilon = \log C + p\log h.
$$

Therefore the slope of a log-log plot of error versus step size estimates the observed order $p$. Equivalently, for two successive step sizes,

$$
p_{\rm obs} =
\frac{\log(\epsilon_2/\epsilon_1)}
{\log(h_2/h_1)}.
$$

A convergence plot is not decorative—it is evidence that the implementation is behaving like the method you think you implemented.


## In-class Activity 2 — Convergence study (15 minutes)

For each method, you will complete only the numerical-integration line. The rest of the cell is already set up to compare the final numerical state $(x,v)$ with the analytic benchmark and store the error in a pandas `DataFrame`.

Use the state-vector error

$$
\epsilon=\sqrt{(x-x_\mathrm{exact})^2+(v-v_\mathrm{exact})^2}.
$$

Your task in the code cell is to compute `t_num` and `y_num` for each method and step size. After that cell is completed, the next worked cells will make the log-log convergence plot and estimate the observed order $p$.


In [ ]:
h_values = np.array([0.025, 0.0125, 0.00625, 0.003125, 0.0015625])
methods_for_convergence = {
    "Euler": euler_step,
    "Midpoint/RK2": midpoint_step,
    "RK4": rk4_step_ref,
}

x_tf, v_tf = oscillator_exact(np.array([t_span[1]]))
y_tf_exact = np.array([x_tf[0], v_tf[0]])

rows = []
for method_name, stepper in methods_for_convergence.items():
    for h in h_values:
        # TODO: integrate with this method and step size.
        # Hint: call integrate_fixed_step(oscillator_rhs, stepper, t_span, y0, h).
        t_num, y_num = ...

        # Compute the Euclidean error of the final state.
        error = np.linalg.norm(y_num[-1] - y_tf_exact)

        rows.append({"method": method_name, "h": h, "error": error})

convergence_df = pd.DataFrame(rows)
convergence_df


In [ ]:
# Worked follow-up: plot the convergence study on log-log axes.
fig, ax = plt.subplots()
for method_name, group in convergence_df.groupby("method"):
    # Sorting by h makes the line connect points in physical step-size order.
    group_sorted = group.sort_values("h")
    ax.loglog(group_sorted["h"], group_sorted["error"], "o-", label=method_name)

ax.set(
    xlabel="step size h [s]",
    ylabel="final-state error",
    title="Observed convergence from a step-size study",
)
ax.legend()
plt.show()


### Worked follow-up: estimate the observed order $p$

On a log-log plot, the model $\epsilon(h)\approx C h^p$ becomes a straight line:

$$
\log \epsilon = \log C + p\log h.
$$

Therefore, the slope of `log(error)` versus `log(h)` estimates the observed order $p$. If the implementations and step sizes are behaving as expected, the slopes should be close to 1 for Euler, 2 for midpoint/RK2, and 4 for RK4 before roundoff effects become important.


In [ ]:
# Worked follow-up: fit one log-log slope for each method.
order_rows = []
for method_name, group in convergence_df.groupby("method"):
    group_sorted = group.sort_values("h")

    # np.polyfit returns [slope, intercept] for a first-degree polynomial.
    observed_order = np.polyfit(
        np.log(group_sorted["h"]),
        np.log(group_sorted["error"]),
        deg=1,
    )[0]

    order_rows.append({
        "method": method_name,
        "observed_order_p": observed_order,
    })

order_df = pd.DataFrame(order_rows)
order_df


## 7. Production workflow: when to use `solve_ivp`

Hand-written fixed-step methods are invaluable for understanding error and convergence. For research code, you will often use an adaptive solver such as `scipy.integrate.solve_ivp` and control the requested accuracy with `rtol` and `atol`.

For most `solve_ivp` methods, the solver estimates the **local error** in each component of the state vector and compares it with a scale of the form

$$
\mathrm{allowed\ error}_i \sim \mathrm{atol}_i + \mathrm{rtol}\, |y_i|.
$$

Here $y_i$ is one component of the numerical state, such as position or velocity.

- `rtol` is the **relative tolerance**. It asks the solver to keep errors small compared with the size of the solution component. Smaller `rtol` usually means more trusted relative digits, smaller internal steps, and more function evaluations.
- `atol` is the **absolute tolerance**. It sets an error floor for components that are near zero. This matters because a purely relative tolerance becomes ambiguous when $y_i \approx 0$.
- For a vector state such as $y=[x,v]$, one scalar `atol` is only appropriate if the components have comparable numerical scales. If position and velocity have very different units or magnitudes, pass an array such as `atol=[atol_x, atol_v]`.

This is different from the fixed-step RK4 method prepared earlier. In fixed-step RK4, you choose a step size $h$ before the integration starts, and the method uses that same step size unless you manually change it. To check accuracy, you rerun the calculation with smaller $h$ and look for convergence. In adaptive `solve_ivp` methods, you choose error tolerances, and the solver changes its internal step size to satisfy those tolerances. The requested output points through `t_eval` are only output locations; they are not necessarily the solver's internal steps.

That does **not** remove the need for validation. A robust workflow is:

1. test the solver on a case with a known solution;
2. inspect solver success/status information;
3. tighten tolerances and check that the result is insensitive;
4. monitor physical diagnostics;
5. retain enough metadata (method, tolerances, interval, parameters) to reproduce the run.


In [ ]:
# Compare fixed-step RK4 with adaptive solve_ivp tolerance choices.
comparison_rows = []

# Fixed-step RK4: accuracy is controlled by manually choosing h.
rk4_h_values = [0.2, 0.1, 0.05, 0.025]
for h in rk4_h_values:
    t_num, y_num = integrate_fixed_step(
        oscillator_rhs, rk4_step_ref, t_span, y0, h=h
    )
    endpoint_error = np.linalg.norm(y_num[-1] - y_tf_exact)
    comparison_rows.append({
        "approach": "fixed RK4",
        "setting": f"h={h:g}",
        "rtol": np.nan,
        "atol": np.nan,
        "steps_or_accepted_steps": len(t_num) - 1,
        "function_evaluations": 4 * (len(t_num) - 1),
        "endpoint_error": endpoint_error,
        "success": True,
    })

# Adaptive RK45 through solve_ivp: accuracy is controlled by tolerances.
tolerances = [1e-4, 1e-7, 1e-10]
for rtol in tolerances:
    atol = rtol * 1e-2
    sol = solve_ivp(
        oscillator_rhs,
        t_span,
        y0,
        method="RK45",
        rtol=rtol,
        atol=atol,
    )
    endpoint_error = np.linalg.norm(sol.y[:, -1] - y_tf_exact)
    comparison_rows.append({
        "approach": "solve_ivp RK45",
        "setting": f"rtol={rtol:g}, atol={atol:g}",
        "rtol": rtol,
        "atol": atol,
        "steps_or_accepted_steps": sol.t.size - 1,
        "function_evaluations": sol.nfev,
        "endpoint_error": endpoint_error,
        "success": sol.success,
    })

solver_comparison_df = pd.DataFrame(comparison_rows)
solver_comparison_df


Common `solve_ivp` methods are summarized below. See the SciPy `solve_ivp` documentation in the references for the full API and method details.

| Method | Type | Good first use case | Possible advantages | Possible cautions |
|---|---|---|---|---|
| `RK45` | Explicit adaptive Runge-Kutta | Default choice for many non-stiff problems | Good general-purpose starting point; automatic step-size control | Can be inefficient or fail for stiff systems |
| `RK23` | Explicit adaptive Runge-Kutta | Low-accuracy exploratory runs | Often cheaper per step than higher-order methods | Usually less efficient when high accuracy is needed |
| `DOP853` | Explicit high-order adaptive Runge-Kutta | Smooth non-stiff problems needing high precision | High order can be very efficient for strict tolerances | More expensive per step; still not intended for stiff dynamics |
| `Radau` | Implicit Runge-Kutta | Stiff problems with rapidly decaying transients | Designed for many stiff systems | Each step solves nonlinear algebraic equations; may require a Jacobian for speed |
| `BDF` | Implicit multistep | Stiff dissipative systems | Efficient for many large stiff problems after startup | Less natural for highly oscillatory non-stiff systems; benefits from Jacobian information |
| `LSODA` | Automatic Adams/BDF switching | Problems where stiffness may appear or disappear | Automatically switches between non-stiff and stiff strategies | Older ODEPACK interface; fewer Python-level options than some modern methods |


## Research-workflow checkpoint

If this oscillator were part of a larger research calculation, record at least:

- the physical parameters and units;
- the initial conditions;
- the solver and version of the code;
- fixed step size or adaptive tolerances;
- the convergence/tolerance study used to justify the numerical settings;
- the physical diagnostic used for validation.

**Assignment 3 connection:** you will repeat this logic on a numerical IVP and a finite-difference BVP: vary resolution, quantify convergence, compare with a benchmark, and interpret the result physically.


## Takeaways

- The correct damped-oscillator benchmark must satisfy *both* stated initial conditions.
- A convergence plot provides quantitative evidence that an implementation behaves with the expected order.
- Physical diagnostics such as energy can expose failure that a trajectory plot hides.
- Library solvers such as `solve_ivp` are powerful, but they still require tolerance studies and physics-based validation.


## Open-source references and further reading

- SciPy `solve_ivp` documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html
- *Fundamentals of Numerical Computation* (Driscoll & Braun), open online text: https://fncbook.com/
- SciPy integration user guide: https://docs.scipy.org/doc/scipy/tutorial/integrate.html
- NumPy documentation: https://numpy.org/doc/

For further study, look up **adaptive embedded Runge–Kutta pairs** and **stiff ODEs**. Those topics explain why production solvers offer several methods rather than a single universal integrator.
